# B3 — equilibrium-augmented: SHORT real-training pilot (picking attempt 3's settings)

**Follow-up paper, Professor Rabczuk's items 2/3.** Attempt 2 (`w_equilibrium=0.12`, the production-scale-measured balance point) gave a real, measured improvement over attempt 1 -- energy 181.8%→103.7%, reaction 216.9%→165.8%, region Cauchy field 1013.8%→550.9% -- but is still roughly 300x worse than the DEM baseline (0.3%/3.0%).

**This is NOT a fixed-batch diagnostic** (unlike every other calibration check in this project so far) -- it is a SHORT (4,000 iterations instead of 50,000) but otherwise REAL training pilot, with the normal randomly-varying material/load batches, because the open question now (does real training accuracy actually improve) is specifically the one thing a fixed-batch check cannot answer. No final FEM evaluation is run here (that is reserved for whichever setting gets a full 50,000-iteration run) -- just comparing the raw `energy` trajectory across 4 variants under identical conditions.

**Why this is needed**: a CPU-only comparison (this session) found no measurable difference between candidate weights/`n_virtual` values on a TOY model over 400 real-batch iterations, but the true PRODUCTION-size model is too slow on CPU to check this directly (a background run did not finish even 150 iterations in 30 minutes). This notebook answers the same question for real, on GPU, cheaply (~4 short runs instead of a full 50,000-iteration commitment) before spending another ~2.3 GPU-hours on a single guess.

**The four variants** (same model/sampling/code path as the real attempt 2 run, each writing to its own throwaway diagnostic directory):
1. `w_equilibrium=0.0` -- pure energy (what `train_B3.py` already does), as the reference point for what a healthy trajectory looks like at 4,000 iterations for this exact architecture/LR.
2. `w_equilibrium=0.05, n_virtual=8` -- lower weight than attempt 2.
3. `w_equilibrium=0.12, n_virtual=32` -- attempt 2's own weight, more virtual test fields (reduces the residual's measured ~±40% per-call sampling noise).
4. `w_equilibrium=0.03, n_virtual=32` -- both levers combined.

**What to send back**: the full printed output, especially the final per-variant "last-500-iter mean energy" summary line at the end of each run -- whichever variant tracks closest to the pure-energy reference is the best candidate for a full 50,000-iteration attempt 3.

In [ ]:
# =====================================================================
#  CELL -- B3 equilibrium-augmented: SHORT (4,000-iteration) real-
#  training pilot comparing 4 candidate settings against a pure-energy
#  reference, before committing to another full 50,000-iteration run.
#  See the markdown cell above for why this is a real-training pilot,
#  not a fixed-batch diagnostic.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

from omar_pfem.train_B3_equilibrium_augmented import get_eq_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')
R = '/content/drive/MyDrive/pfem_run'
N_ITERS = 4000

VARIANTS = [
    ('baseline_w0.0',      0.0,  8),
    ('w0.05_nv8',          0.05, 8),
    ('w0.12_nv32',         0.12, 32),
    ('w0.03_nv32',         0.03, 32),
]

n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nPilot: {DEFAULT_RESOLUTION} -> {n_elem} elements, {N_ITERS} iterations per variant, '
      f'{len(VARIANTS)} variants, no final FEM evaluation (reserved for the chosen full run).')

summary = []
for name, w_eq, n_virtual in VARIANTS:
    print(f'\n{"=" * 90}')
    print(f'=== Variant: {name} (w_equilibrium={w_eq}, n_virtual={n_virtual}) ===')
    OUTPUT_DIR = f'{R}/b3_equilibrium_diag_{name}'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    args = get_eq_args([
        '--n_iters', str(N_ITERS),
        '--batch_size', '8',
        '--log_every', '250',
        '--ckpt_every', str(N_ITERS),
        '--output_dir', OUTPUT_DIR,
        '--w_equilibrium', str(w_eq),
        '--n_virtual', str(n_virtual),
    ])
    torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    model = train(args, device)
    elapsed = time.time() - t0
    print(f'{name}: finished in {elapsed:.1f}s ({elapsed/N_ITERS:.4f}s/iteration).')
    summary.append((name, w_eq, n_virtual, elapsed))

print(f'\n{"=" * 90}')
print('=== Pilot summary (compare the printed `energy` trend above for each variant; '
      'the baseline (w=0) is the reference -- whichever other variant tracks closest to '
      'it is the best candidate for a full 50,000-iteration attempt 3) ===')
for name, w_eq, n_virtual, elapsed in summary:
    print(f'  {name}: w_equilibrium={w_eq}, n_virtual={n_virtual}, {elapsed:.1f}s total')

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(f'{R}/b3_equilibrium_diag_pilot', kind='b3_equilibrium_augmented_diagnostic',
                    args={'n_iters': N_ITERS, 'variants': [v[0] for v in VARIANTS]},
                    started_at=_started,
                    results={'elapsed_total_s': time.time() - _started},
                    outputs=[f'{R}/b3_equilibrium_diag_{v[0]}' for v in VARIANTS],
                    notes="Short (4,000-iteration) real-training pilot comparing candidate "
                          "w_equilibrium/n_virtual settings for the equilibrium-augmented B3 "
                          "formulation, before committing another full 50,000-iteration run. "
                          "Attempt 2 (w=0.12, nv=8) already improved on attempt 1 but is "
                          "still ~300x worse than the DEM baseline. No final FEM evaluation "
                          "in this pilot -- see B3_Equilibrium_Augmented_Diagnostic.ipynb's "
                          "own markdown cell for the full rationale.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back -- especially the per-variant '
      '`energy` trend at iterations 250/500/.../4000 -- so we can pick attempt 3\'s settings.')
